In [0]:
base_path = "abfss://bronze@sgcustomerpipeline.dfs.core.windows.net/SalesLT/"

table_name = []

for i in dbutils.fs.ls(base_path):
    table_name.append(i.name.split('/')[0])

table_name

In [0]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

for i in table_name:
    path = "abfss://bronze@sgcustomerpipeline.dfs.core.windows.net/SalesLT/" + i + "/" + i + ".parquet"
    df = spark.read.format("parquet").load(path)
    column = df.columns

    for col in column:
        if "Date" in col or "date" in col:
            df = df.withColumn(col, date_format(from_utc_timestamp(df[col].cast(TimestampType()), "UTC"), "yyyy-MM-dd"))
    
    output_path = "abfss://silver@sgcustomerpipeline.dfs.core.windows.net/SalesLT/" + i + "/"
    df.write.format("delta").mode("overwrite").save(output_path)

In [0]:
display(df)